# Building the FrameFinder recommendation model

This notebook walks through the model used by the Streamlit app. Run it from the project directory with the project dependencies installed in a Python 3.12 kernel. The two TMDB source tables are joined by movie ID, not title. Titles are not unique: the dataset contains two versions each of *Batman*, *The Host*, and *Out of the Blue*.

The model is content based. It compares each movie's overview, genres, keywords, three top-billed cast members, and director. Ratings and popularity are kept for display and tie-breaking, but they are not model features.

In [ ]:
import pandas as pd

from build_model import PROJECT_ROOT, prepare_movies, save_artifact, source_file
from recommender import MovieRecommender, vectorize_tags

## Load and prepare the data

The build module resolves paths from its own location, so the notebook uses the same source files as the command-line script.

In [ ]:
project_root = PROJECT_ROOT

data_dir = project_root / 'Data'
movies = prepare_movies(
    source_file(data_dir, 'tmdb_5000_movies.csv'),
    source_file(data_dir, 'tmdb_5000_credits.csv'),
)
movies.head(3)

The raw movie table has 4,803 rows. Three records have no overview and are removed because the synopsis is the strongest text feature. The remaining 4,800 movie IDs must be unique before the sparse feature index is built.

In [ ]:
summary = pd.Series({
    'model_rows': len(movies),
    'unique_movie_ids': movies['movie_id'].nunique(),
    'duplicate_titles': movies['title'].duplicated(keep=False).sum(),
    'missing_overviews': movies['overview'].isna().sum(),
})
assert summary['model_rows'] == summary['unique_movie_ids'] == 4_800
summary

In [ ]:
movies.loc[
    movies['title'].duplicated(keep=False),
    ['movie_id', 'title', 'year'],
].sort_values(['title', 'year'])

## Build TF-IDF features

`prepare_movies` gives extra weight to structured metadata by repeating genre, keyword, cast, and director tokens. `TfidfVectorizer` then reduces the influence of words that appear across many films. The sparse feature index is kept in memory, and cosine similarity is calculated only for the selected movie.

In [ ]:
feature_matrix, feature_count = vectorize_tags(movies['tags'])
sparse_bytes = feature_matrix.data.nbytes + feature_matrix.indices.nbytes + feature_matrix.indptr.nbytes
{
    'features': feature_count,
    'matrix_shape': feature_matrix.shape,
    'matrix_dtype': str(feature_matrix.dtype),
    'size_mb': round(sparse_bytes / 1024**2, 1),
}

## Inspect a recommendation

The production class validates the artifacts and handles ranking. Running the same class here keeps the notebook and app behavior in sync.

In [ ]:
recommender = MovieRecommender(movies)
selected_id = int(movies.loc[movies['title'] == 'Batman Begins', 'movie_id'].iloc[0])
recommendations = recommender.recommend(selected_id, limit=5)

pd.DataFrame([
    {
        'rank': item.rank,
        'title': item.title,
        'year': item.year,
        'similarity': round(item.score, 3),
    }
    for item in recommendations
])

## Save the app artifact

The build script runs these same steps from the command line. Saving here is useful after experimenting with preprocessing in the notebook.

In [ ]:
save_artifact(movies, project_root)
print(f'Saved movies_dict.pkl for {len(movies):,} movies to {project_root}')